# AP Invoice Pipeline — Notebook Edition

Automated accounts payable invoice processing using Snowflake AI functions.
PDF invoices land on a stage, **AI_EXTRACT** pulls structured fields,
a **Task + Stream** pipeline validates and routes them,
and a **Streamlit** dashboard provides human review and analytics.

| Tech | Role |
|------|------|
| AI_EXTRACT | Structured field extraction from PDF invoices |
| AI_CLASSIFY | GL account code suggestion from line-item descriptions |
| Stream + Task | Automated validation pipeline triggered by new data |
| Cortex Analyst | Natural-language analytics via semantic view |
| Streamlit | Interactive review queue and dashboard (deployed in Step 10) |

**Run top-to-bottom** to deploy all objects, load sample data, and explore the pipeline.
The final cell deploys the interactive Streamlit dashboard.

## Architecture

```
PDF Invoices → @RAW_INVOICE_STAGE
                    │
                    ▼
            SP_PROCESS_INVOICE
           (AI_EXTRACT → Vendor Match → Score)
                    │
              ┌─────┴─────┐
              ▼            ▼
         Score ≥ 0.75  Score < 0.75
         AUTO-APPROVE  → REVIEW_QUEUE
              │            │
              ▼            ▼
      INVOICE_HEADER   Streamlit Review Panel
      (PROCESSED)      (Human approve/reject)
              │            │
              └─────┬──────┘
                    ▼
          PROCESSED_INVOICES (view)
                    │
                    ▼
          Cortex Analyst (NL queries)
          via SV_AP_INVOICE semantic view
```

## Step 1: Infrastructure Setup

Creates the shared API integration, database, schema, and warehouse.
All statements are idempotent — safe to re-run.

In [1]:
USE ROLE ACCOUNTADMIN;
CREATE API INTEGRATION IF NOT EXISTS SFE_GIT_API_INTEGRATION
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/sfc-gh-miwhitaker/sfe-public')
  ENABLED = TRUE
  COMMENT = 'Shared Git integration for sfe-public monorepo';

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE SYSADMIN;

USE ROLE SYSADMIN;
CREATE DATABASE IF NOT EXISTS SNOWFLAKE_EXAMPLE;

CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_EXAMPLE.AP_INVOICE
  COMMENT = 'AP Invoice automation with AI_EXTRACT and AI_CLASSIFY';

CREATE WAREHOUSE IF NOT EXISTS SFE_AP_INVOICE_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  STATEMENT_TIMEOUT_IN_SECONDS = 300
  COMMENT = 'AP Invoice Pipeline compute';

USE WAREHOUSE SFE_AP_INVOICE_WH;
USE SCHEMA SNOWFLAKE_EXAMPLE.AP_INVOICE;

SyntaxError: invalid syntax (2060016548.py, line 1)

## Step 2: Schema & Tables

Star-like schema with `INVOICE_HEADER` as the central fact table.
Reference tables (`VENDOR_MASTER`, `GL_CODES`) support fuzzy matching and AI classification.
`REVIEW_QUEUE` and `AUDIT_LOG` capture the human-in-the-loop and auditability layers.

| Table | Purpose |
|-------|---------|
| VENDOR_MASTER | Canonical vendor list with alias arrays for fuzzy matching |
| GL_CODES | GL account taxonomy passed to AI_CLASSIFY as category labels |
| INVOICE_HEADER | One row per PDF — extracted fields + validation score |
| INVOICE_LINE_ITEMS | Many rows per invoice with AI-classified GL codes |
| REVIEW_QUEUE | Low-scoring invoices awaiting human review |
| AUDIT_LOG | Immutable record of every AI and human decision |

In [ ]:
CREATE STAGE IF NOT EXISTS RAW_INVOICE_STAGE
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
  COMMENT = 'Landing zone for PDF invoices';

CREATE TABLE IF NOT EXISTS VENDOR_MASTER (
    VENDOR_ID        NUMBER AUTOINCREMENT PRIMARY KEY,
    VENDOR_NAME      VARCHAR NOT NULL,
    VENDOR_ALIASES   ARRAY,
    PAYMENT_TERMS    VARCHAR DEFAULT 'NET30'
);

CREATE TABLE IF NOT EXISTS GL_CODES (
    GL_CODE          VARCHAR(10) PRIMARY KEY,
    GL_DESCRIPTION   VARCHAR NOT NULL,
    CATEGORY         VARCHAR NOT NULL
);

CREATE TABLE IF NOT EXISTS INVOICE_HEADER (
    INVOICE_ID          NUMBER AUTOINCREMENT PRIMARY KEY,
    SOURCE_FILE         VARCHAR NOT NULL,
    VENDOR_NAME_RAW     VARCHAR,
    VENDOR_ID_RESOLVED  NUMBER,
    INVOICE_NUMBER      VARCHAR,
    INVOICE_DATE        DATE,
    PO_REFERENCE        VARCHAR,
    TOTAL_AMOUNT        NUMBER(12,2),
    CURRENCY            VARCHAR(3) DEFAULT 'USD',
    PROPERTY            VARCHAR,
    EXTRACTION_TS       TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    VALIDATION_SCORE    NUMBER(5,2),
    STATUS              VARCHAR DEFAULT 'PENDING',
    AI_EXTRACT_RAW      VARIANT,
    APPROVED_BY         VARCHAR,
    APPROVED_TS         TIMESTAMP_NTZ
);

CREATE TABLE IF NOT EXISTS INVOICE_LINE_ITEMS (
    LINE_ID              NUMBER AUTOINCREMENT PRIMARY KEY,
    INVOICE_ID           NUMBER NOT NULL,
    DESCRIPTION          VARCHAR,
    QUANTITY             NUMBER(10,2),
    UNIT_PRICE           NUMBER(12,2),
    LINE_TOTAL           NUMBER(12,2),
    GL_CODE_SUGGESTED    VARCHAR(10),
    GL_CODE_CONFIDENCE   NUMBER(5,4),
    GL_CODE_CONFIRMED    VARCHAR(10),
    REVIEWER_OVERRIDE    BOOLEAN DEFAULT FALSE
);

CREATE TABLE IF NOT EXISTS REVIEW_QUEUE (
    QUEUE_ID            NUMBER AUTOINCREMENT PRIMARY KEY,
    INVOICE_ID          NUMBER NOT NULL,
    FLAGGED_FIELDS      ARRAY,
    VALIDATION_SCORE    NUMBER(5,2),
    REVIEWER_ID         VARCHAR,
    REVIEWED_TS         TIMESTAMP_NTZ,
    RESOLUTION          VARCHAR,
    NOTES               VARCHAR
);

CREATE TABLE IF NOT EXISTS AUDIT_LOG (
    LOG_ID              NUMBER AUTOINCREMENT PRIMARY KEY,
    INVOICE_ID          NUMBER NOT NULL,
    ACTION              VARCHAR NOT NULL,
    FIELD_NAME          VARCHAR,
    OLD_VALUE           VARCHAR,
    NEW_VALUE           VARCHAR,
    ACTOR               VARCHAR NOT NULL,
    ACTOR_TYPE          VARCHAR NOT NULL,
    ACTION_TS           TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

## Step 3: Sample Data

Loads realistic synthetic data: 8 vendors, 20 GL codes, 27 invoices (23 clean + 4 ambiguous),
line items, review queue entries, and audit log records.

In [ ]:
-- Vendor Master (8 vendors across hospitality categories)
INSERT INTO VENDOR_MASTER (VENDOR_NAME, VENDOR_ALIASES, PAYMENT_TERMS)
SELECT column1, PARSE_JSON(column2), column3
FROM VALUES
    ('Alpine Food Supply Co.',       '["Alpine Foods","Alpine Food Svc","AFS Co"]',           'NET30'),
    ('Continental Linen Services',   '["Continental Linen","CLS Inc","Cont. Linen"]',         'NET30'),
    ('Metro IT Solutions',           '["Metro IT","MIS Corp","Metro Info Tech"]',              'NET45'),
    ('National Maintenance Group',   '["Natl Maintenance","NMG Services","Nat Maint Group"]', 'NET30'),
    ('Premier Entertainment Inc.',   '["Premier Ent","PEI","Premier Entertainment"]',         'NET15'),
    ('Statewide Beverage Dist.',     '["Statewide Bev","SBD","State Beverage"]',              'NET30'),
    ('Consolidated Building Supply', '["CBS","Consolidated Bldg","Consol Building"]',         'NET45'),
    ('Pacific Seafood Partners',     '["Pacific Seafood","PSP","Pac Seafood"]',               'NET30');

-- GL Code Taxonomy (hospitality-specific)
INSERT INTO GL_CODES (GL_CODE, GL_DESCRIPTION, CATEGORY)
VALUES
    ('5100', 'Food & Beverage',           'Operating'),
    ('5110', 'Dry Goods & Pantry',        'Operating'),
    ('5120', 'Fresh Produce & Dairy',     'Operating'),
    ('5130', 'Beverages & Bar Stock',     'Operating'),
    ('5200', 'Lodging Supplies',          'Operating'),
    ('5210', 'Linens & Bedding',          'Operating'),
    ('5220', 'Guest Amenities',           'Operating'),
    ('5300', 'Entertainment',             'Operating'),
    ('5310', 'Talent & Performers',       'Operating'),
    ('5320', 'AV & Production',           'Operating'),
    ('5400', 'Facilities & Maintenance',  'Operating'),
    ('5410', 'HVAC & Plumbing',           'Operating'),
    ('5420', 'Electrical & Lighting',     'Operating'),
    ('5430', 'Building Materials',        'Operating'),
    ('5500', 'IT Services',              'Operating'),
    ('5510', 'Software & Licensing',      'Operating'),
    ('5520', 'Hardware & Equipment',      'Operating'),
    ('6100', 'Professional Services',     'G&A'),
    ('6200', 'Insurance',                 'G&A'),
    ('6300', 'Utilities',                 'G&A');

In [ ]:
-- Clean invoices (23) — high validation scores, auto-approved
INSERT INTO INVOICE_HEADER
    (SOURCE_FILE, VENDOR_NAME_RAW, VENDOR_ID_RESOLVED, INVOICE_NUMBER, INVOICE_DATE,
     PO_REFERENCE, TOTAL_AMOUNT, CURRENCY, PROPERTY, VALIDATION_SCORE, STATUS, APPROVED_BY, APPROVED_TS)
SELECT column1, column2, column3, column4, column5::DATE,
       column6, column7::NUMBER(12,2), column8, column9, column10::NUMBER(5,2), column11, column12, column13::TIMESTAMP_NTZ
FROM VALUES
    ('inv_alpine_001.pdf',      'Alpine Food Supply Co.',     1, 'AFS-2026-0441', '2026-03-01', 'PO-8801', 12450.00, 'USD', 'Resort East',      0.95, 'PROCESSED', 'SYSTEM', '2026-03-01 09:15:00'),
    ('inv_alpine_002.pdf',      'Alpine Food Supply Co.',     1, 'AFS-2026-0442', '2026-03-08', 'PO-8802', 8920.50,  'USD', 'Resort Northeast',  0.92, 'PROCESSED', 'SYSTEM', '2026-03-08 09:20:00'),
    ('inv_alpine_003.pdf',      'Alpine Food Supply Co.',     1, 'AFS-2026-0443', '2026-03-15', 'PO-8810', 15680.00, 'USD', 'Resort East',      0.97, 'PROCESSED', 'SYSTEM', '2026-03-15 09:10:00'),
    ('inv_contin_001.pdf',      'Continental Linen Services', 2, 'CLS-44021',     '2026-03-03', 'PO-8803', 6340.00,  'USD', 'Resort East',      0.91, 'PROCESSED', 'SYSTEM', '2026-03-03 10:00:00'),
    ('inv_contin_002.pdf',      'Continental Linen Services', 2, 'CLS-44022',     '2026-03-10', 'PO-8807', 7125.00,  'USD', 'Resort North',     0.93, 'PROCESSED', 'SYSTEM', '2026-03-10 10:05:00'),
    ('inv_contin_003.pdf',      'Continental Linen Services', 2, 'CLS-44023',     '2026-03-17', 'PO-8811', 5890.00,  'USD', 'Resort Northeast',  0.90, 'PROCESSED', 'SYSTEM', '2026-03-17 10:10:00'),
    ('inv_metro_001.pdf',       'Metro IT Solutions',         3, 'MIT-2026-1001', '2026-03-05', 'PO-8804', 24500.00, 'USD', 'Resort East',      0.96, 'PROCESSED', 'SYSTEM', '2026-03-05 11:00:00'),
    ('inv_metro_002.pdf',       'Metro IT Solutions',         3, 'MIT-2026-1002', '2026-03-12', NULL,       18750.00, 'USD', 'Resort Northeast',  0.88, 'PROCESSED', 'SYSTEM', '2026-03-12 11:05:00'),
    ('inv_national_001.pdf',    'National Maintenance Group', 4, 'NMG-60110',     '2026-03-02', 'PO-8805', 9870.00,  'USD', 'Resort East',      0.94, 'PROCESSED', 'SYSTEM', '2026-03-02 08:30:00'),
    ('inv_national_002.pdf',    'National Maintenance Group', 4, 'NMG-60111',     '2026-03-09', 'PO-8806', 11200.00, 'USD', 'Resort North',     0.91, 'PROCESSED', 'SYSTEM', '2026-03-09 08:35:00'),
    ('inv_national_003.pdf',    'National Maintenance Group', 4, 'NMG-60112',     '2026-03-20', 'PO-8815', 7650.00,  'USD', 'Resort Northeast',  0.93, 'PROCESSED', 'SYSTEM', '2026-03-20 08:40:00'),
    ('inv_premier_001.pdf',     'Premier Entertainment Inc.', 5, 'PEI-9900',      '2026-03-04', 'PO-8808', 35000.00, 'USD', 'Resort East',      0.95, 'PROCESSED', 'SYSTEM', '2026-03-04 14:00:00'),
    ('inv_premier_002.pdf',     'Premier Entertainment Inc.', 5, 'PEI-9901',      '2026-03-18', 'PO-8812', 42500.00, 'USD', 'Resort Northeast',  0.94, 'PROCESSED', 'SYSTEM', '2026-03-18 14:05:00'),
    ('inv_statewide_001.pdf',   'Statewide Beverage Dist.',   6, 'SBD-2026-771',  '2026-03-06', 'PO-8809', 16800.00, 'USD', 'Resort East',      0.96, 'PROCESSED', 'SYSTEM', '2026-03-06 07:45:00'),
    ('inv_statewide_002.pdf',   'Statewide Beverage Dist.',   6, 'SBD-2026-772',  '2026-03-13', 'PO-8813', 14350.00, 'USD', 'Resort North',     0.92, 'PROCESSED', 'SYSTEM', '2026-03-13 07:50:00'),
    ('inv_statewide_003.pdf',   'Statewide Beverage Dist.',   6, 'SBD-2026-773',  '2026-03-22', 'PO-8816', 19200.00, 'USD', 'Resort Northeast',  0.94, 'PROCESSED', 'SYSTEM', '2026-03-22 07:55:00'),
    ('inv_consol_001.pdf',      'Consolidated Building Supply', 7, 'CBS-30440',   '2026-03-07', 'PO-8814', 8900.00,  'USD', 'Resort East',      0.90, 'PROCESSED', 'SYSTEM', '2026-03-07 12:00:00'),
    ('inv_consol_002.pdf',      'Consolidated Building Supply', 7, 'CBS-30441',   '2026-03-14', NULL,       6750.00,  'USD', 'Resort North',     0.87, 'PROCESSED', 'SYSTEM', '2026-03-14 12:05:00'),
    ('inv_pacific_001.pdf',     'Pacific Seafood Partners',   8, 'PSP-2026-088',  '2026-03-11', 'PO-8817', 11400.00, 'USD', 'Resort East',      0.95, 'PROCESSED', 'SYSTEM', '2026-03-11 06:30:00'),
    ('inv_pacific_002.pdf',     'Pacific Seafood Partners',   8, 'PSP-2026-089',  '2026-03-19', 'PO-8818', 9850.00,  'USD', 'Resort Northeast',  0.93, 'PROCESSED', 'SYSTEM', '2026-03-19 06:35:00'),
    ('inv_alpine_004.pdf',      'Alpine Food Supply Co.',     1, 'AFS-2026-0444', '2026-03-25', 'PO-8820', 10200.00, 'USD', 'Resort North',     0.91, 'PROCESSED', 'SYSTEM', '2026-03-25 09:25:00'),
    ('inv_national_004.pdf',    'National Maintenance Group', 4, 'NMG-60113',     '2026-03-28', 'PO-8822', 13500.00, 'USD', 'Resort East',      0.92, 'PROCESSED', 'SYSTEM', '2026-03-28 08:45:00'),
    ('inv_contin_004.pdf',      'Continental Linen Services', 2, 'CLS-44024',     '2026-03-30', 'PO-8824', 8100.00,  'USD', 'Resort East',      0.94, 'PROCESSED', 'SYSTEM', '2026-03-30 10:15:00');

-- Ambiguous invoices (4) — low validation scores, routed to review queue
INSERT INTO INVOICE_HEADER
    (SOURCE_FILE, VENDOR_NAME_RAW, VENDOR_ID_RESOLVED, INVOICE_NUMBER, INVOICE_DATE,
     PO_REFERENCE, TOTAL_AMOUNT, CURRENCY, PROPERTY, VALIDATION_SCORE, STATUS)
SELECT column1, column2, column3, column4, column5::DATE,
       column6, column7::NUMBER(12,2), column8, column9, column10::NUMBER(5,2), column11
FROM VALUES
    ('inv_unknown_001.pdf',  'Alpine Foods LLC',     NULL, 'AF-99X',      '2026-03-21', NULL,       4580.00,  'USD', 'Resort East',     0.52, 'REVIEW'),
    ('inv_unclear_002.pdf',  'Metro Info Tech',      NULL, NULL,          '2026-03-23', 'PO-DRAFT', 31200.00, 'USD', 'Resort Northeast', 0.41, 'REVIEW'),
    ('inv_partial_003.pdf',  'Natl Maintenance',     NULL, 'NMG-PARTIAL', NULL,         NULL,       NULL,     'USD', 'Resort North',     0.28, 'REVIEW'),
    ('inv_damaged_004.pdf',  'State Beverage',       NULL, 'SBD-SCAN',    '2026-03-26', NULL,       7200.00,  'USD', 'Resort East',     0.63, 'REVIEW');

In [ ]:
-- Review Queue entries for ambiguous invoices
INSERT INTO REVIEW_QUEUE (INVOICE_ID, FLAGGED_FIELDS, VALIDATION_SCORE, RESOLUTION)
SELECT column1, PARSE_JSON(column2), column3::NUMBER(5,2), column4
FROM VALUES
    (24, '["VENDOR_ID_RESOLVED","PO_REFERENCE"]',                        0.52, NULL),
    (25, '["VENDOR_ID_RESOLVED","INVOICE_NUMBER","PO_REFERENCE"]',       0.41, NULL),
    (26, '["VENDOR_ID_RESOLVED","INVOICE_DATE","TOTAL_AMOUNT"]',         0.28, NULL),
    (27, '["VENDOR_ID_RESOLVED","PO_REFERENCE"]',                        0.63, NULL);

-- Line Items for processed invoices (representative subset)
INSERT INTO INVOICE_LINE_ITEMS
    (INVOICE_ID, DESCRIPTION, QUANTITY, UNIT_PRICE, LINE_TOTAL,
     GL_CODE_SUGGESTED, GL_CODE_CONFIDENCE, GL_CODE_CONFIRMED, REVIEWER_OVERRIDE)
SELECT column1, column2, column3::NUMBER(10,2), column4::NUMBER(12,2), column5::NUMBER(12,2),
       column6, column7::NUMBER(5,4), column8, column9::BOOLEAN
FROM VALUES
    (1, 'Dry goods pantry restock - flour, sugar, spices',   1, 4200.00, 4200.00, '5110', 0.9200, '5110', FALSE),
    (1, 'Fresh produce weekly delivery',                     1, 5100.00, 5100.00, '5120', 0.9500, '5120', FALSE),
    (1, 'Dairy products - milk, cream, butter',              1, 3150.00, 3150.00, '5120', 0.9100, '5120', FALSE),
    (4, 'King size sheet sets (200 count)',                 200, 18.00,   3600.00, '5210', 0.9400, '5210', FALSE),
    (4, 'Bath towel replacement lot',                      400,  4.50,   1800.00, '5210', 0.9300, '5210', FALSE),
    (4, 'Pillow protectors',                               300,  3.13,    940.00, '5210', 0.8900, '5210', FALSE),
    (7, 'Managed network services - March',                  1, 15000.00, 15000.00, '5500', 0.9600, '5500', FALSE),
    (7, 'Software license renewal - POS system',             1,  6500.00,  6500.00, '5510', 0.9100, '5510', FALSE),
    (7, 'Replacement access points (qty 12)',               12,   250.00,  3000.00, '5520', 0.8800, '5520', FALSE),
    (9,  'HVAC quarterly maintenance - main building',       1,  4200.00,  4200.00, '5410', 0.9300, '5410', FALSE),
    (9,  'Emergency plumbing repair - 3rd floor',            1,  2870.00,  2870.00, '5410', 0.8700, '5410', FALSE),
    (9,  'Light fixture replacement - lobby',               20,   140.00,  2800.00, '5420', 0.9000, '5420', FALSE),
    (12, 'Headline performer - March 15 weekend',            1, 25000.00, 25000.00, '5310', 0.9500, '5310', FALSE),
    (12, 'Sound & lighting production package',              1,  7500.00,  7500.00, '5320', 0.9200, '5320', FALSE),
    (12, 'Backstage catering & hospitality',                 1,  2500.00,  2500.00, '5100', 0.7800, '5100', FALSE),
    (14, 'Premium spirits replenishment',                    1,  8400.00,  8400.00, '5130', 0.9400, '5130', FALSE),
    (14, 'Draft beer kegs (assorted)',                      40,   120.00,  4800.00, '5130', 0.9600, '5130', FALSE),
    (14, 'Non-alcoholic beverage cases',                   200,    18.00,  3600.00, '5130', 0.9100, '5130', FALSE),
    (19, 'Fresh Atlantic salmon filets (200 lbs)',           1,  4800.00,  4800.00, '5120', 0.9300, '5120', FALSE),
    (19, 'Lobster tails for weekend service',              100,    42.00,  4200.00, '5120', 0.9500, '5120', FALSE),
    (19, 'Shrimp & shellfish assortment',                    1,  2400.00,  2400.00, '5120', 0.9100, '5120', FALSE),
    (24, 'Food supplies - misc',                             1,  2580.00,  2580.00, '5100', 0.6200, NULL,   FALSE),
    (24, 'Delivery surcharge',                               1,  2000.00,  2000.00, '6100', 0.4500, NULL,   FALSE),
    (25, 'Annual IT infrastructure assessment',              1, 18000.00, 18000.00, '5500', 0.7100, NULL,   FALSE),
    (25, 'Cybersecurity audit - all properties',             1, 13200.00, 13200.00, '5500', 0.6800, NULL,   FALSE);

-- Audit Log entries for processed invoices
INSERT INTO AUDIT_LOG (INVOICE_ID, ACTION, FIELD_NAME, OLD_VALUE, NEW_VALUE, ACTOR, ACTOR_TYPE, ACTION_TS)
SELECT column1, column2, column3, column4, column5, column6, column7, column8::TIMESTAMP_NTZ
FROM VALUES
    (1,  'EXTRACTED',      NULL,               NULL,    NULL,                'AI_EXTRACT',  'AI',    '2026-03-01 09:10:00'),
    (1,  'VENDOR_MATCHED', 'VENDOR_ID_RESOLVED', NULL,  '1',                'FUZZY_MATCH', 'AI',    '2026-03-01 09:11:00'),
    (1,  'GL_CLASSIFIED',  'GL_CODE_SUGGESTED',  NULL,  '5110,5120,5120',   'AI_CLASSIFY', 'AI',    '2026-03-01 09:12:00'),
    (1,  'AUTO_APPROVED',  'STATUS',           'PENDING','PROCESSED',        'SYSTEM',      'SYSTEM','2026-03-01 09:15:00'),
    (24, 'EXTRACTED',      NULL,               NULL,    NULL,                'AI_EXTRACT',  'AI',    '2026-03-21 09:00:00'),
    (24, 'VENDOR_UNMATCHED','VENDOR_ID_RESOLVED',NULL,  NULL,               'FUZZY_MATCH', 'AI',    '2026-03-21 09:01:00'),
    (24, 'FLAGGED',        'STATUS',           'PENDING','REVIEW',           'SYSTEM',      'SYSTEM','2026-03-21 09:02:00'),
    (25, 'EXTRACTED',      NULL,               NULL,    NULL,                'AI_EXTRACT',  'AI',    '2026-03-23 10:00:00'),
    (25, 'FLAGGED',        'STATUS',           'PENDING','REVIEW',           'SYSTEM',      'SYSTEM','2026-03-23 10:01:00'),
    (26, 'EXTRACTED',      NULL,               NULL,    NULL,                'AI_EXTRACT',  'AI',    '2026-03-24 11:00:00'),
    (26, 'FLAGGED',        'STATUS',           'PENDING','REVIEW',           'SYSTEM',      'SYSTEM','2026-03-24 11:01:00');

In [ ]:
SELECT
    (SELECT COUNT(*) FROM VENDOR_MASTER)       AS vendors,
    (SELECT COUNT(*) FROM GL_CODES)            AS gl_codes,
    (SELECT COUNT(*) FROM INVOICE_HEADER)      AS invoices,
    (SELECT COUNT(*) FROM INVOICE_LINE_ITEMS)  AS line_items,
    (SELECT COUNT(*) FROM REVIEW_QUEUE)        AS review_queue,
    (SELECT COUNT(*) FROM AUDIT_LOG)           AS audit_entries;

## Step 4: Analytical Views

Five views layer analytics on top of the base tables:

| View | Purpose |
|------|---------|
| `PROCESSED_INVOICES` | Joined invoice data with vendor resolution and line-item totals |
| `V_REVIEW_QUEUE` | Pending review items with full invoice context |
| `V_LINE_ITEMS_ENRICHED` | Line items with GL code descriptions and classification status |
| `V_PIPELINE_METRICS` | Aggregated pipeline KPIs (counts, rates, averages) |
| `V_SPEND_ANALYSIS` | Spend breakdown by property, vendor, and GL category |

In [ ]:
CREATE OR REPLACE VIEW PROCESSED_INVOICES AS
SELECT
    h.INVOICE_ID,
    h.SOURCE_FILE,
    h.INVOICE_NUMBER,
    h.INVOICE_DATE,
    h.PO_REFERENCE,
    h.TOTAL_AMOUNT,
    h.CURRENCY,
    h.PROPERTY,
    h.VALIDATION_SCORE,
    h.STATUS,
    h.EXTRACTION_TS,
    h.APPROVED_BY,
    h.APPROVED_TS,
    DATEDIFF('second', h.EXTRACTION_TS, COALESCE(h.APPROVED_TS, CURRENT_TIMESTAMP())) AS PROCESSING_SECONDS,
    h.VENDOR_NAME_RAW,
    v.VENDOR_NAME      AS VENDOR_NAME_RESOLVED,
    v.VENDOR_ID        AS VENDOR_ID,
    v.PAYMENT_TERMS,
    li.LINE_COUNT,
    li.LINE_TOTAL_SUM
FROM INVOICE_HEADER h
LEFT JOIN VENDOR_MASTER v
    ON h.VENDOR_ID_RESOLVED = v.VENDOR_ID
LEFT JOIN (
    SELECT
        INVOICE_ID,
        COUNT(*)         AS LINE_COUNT,
        SUM(LINE_TOTAL)  AS LINE_TOTAL_SUM
    FROM INVOICE_LINE_ITEMS
    GROUP BY INVOICE_ID
) li
    ON h.INVOICE_ID = li.INVOICE_ID;

CREATE OR REPLACE VIEW V_REVIEW_QUEUE AS
SELECT
    rq.QUEUE_ID,
    rq.INVOICE_ID,
    h.SOURCE_FILE,
    h.VENDOR_NAME_RAW,
    h.INVOICE_NUMBER,
    h.INVOICE_DATE,
    h.TOTAL_AMOUNT,
    h.PROPERTY,
    rq.FLAGGED_FIELDS,
    rq.VALIDATION_SCORE,
    rq.REVIEWER_ID,
    rq.REVIEWED_TS,
    rq.RESOLUTION,
    rq.NOTES
FROM REVIEW_QUEUE rq
JOIN INVOICE_HEADER h
    ON rq.INVOICE_ID = h.INVOICE_ID;

CREATE OR REPLACE VIEW V_LINE_ITEMS_ENRICHED AS
SELECT
    li.LINE_ID,
    li.INVOICE_ID,
    h.INVOICE_NUMBER,
    h.PROPERTY,
    li.DESCRIPTION,
    li.QUANTITY,
    li.UNIT_PRICE,
    li.LINE_TOTAL,
    li.GL_CODE_SUGGESTED,
    g_suggested.GL_DESCRIPTION  AS GL_SUGGESTED_DESC,
    g_suggested.CATEGORY        AS GL_SUGGESTED_CATEGORY,
    li.GL_CODE_CONFIDENCE,
    li.GL_CODE_CONFIRMED,
    g_confirmed.GL_DESCRIPTION  AS GL_CONFIRMED_DESC,
    li.REVIEWER_OVERRIDE,
    CASE
        WHEN li.GL_CODE_CONFIRMED IS NOT NULL THEN 'Human Confirmed'
        WHEN li.GL_CODE_CONFIDENCE >= 0.85    THEN 'AI Suggested (High)'
        WHEN li.GL_CODE_CONFIDENCE >= 0.70    THEN 'AI Suggested (Medium)'
        ELSE 'Needs Review'
    END AS CLASSIFICATION_STATUS
FROM INVOICE_LINE_ITEMS li
JOIN INVOICE_HEADER h
    ON li.INVOICE_ID = h.INVOICE_ID
LEFT JOIN GL_CODES g_suggested
    ON li.GL_CODE_SUGGESTED = g_suggested.GL_CODE
LEFT JOIN GL_CODES g_confirmed
    ON li.GL_CODE_CONFIRMED = g_confirmed.GL_CODE;

CREATE OR REPLACE VIEW V_PIPELINE_METRICS AS
SELECT
    COUNT(*)                                                      AS TOTAL_INVOICES,
    COUNT_IF(STATUS = 'PROCESSED')                                AS PROCESSED_COUNT,
    COUNT_IF(STATUS = 'REVIEW')                                   AS REVIEW_COUNT,
    COUNT_IF(STATUS = 'PENDING')                                  AS PENDING_COUNT,
    ROUND(COUNT_IF(STATUS = 'PROCESSED') / NULLIF(COUNT(*), 0) * 100, 1)
                                                                  AS AUTO_APPROVAL_RATE,
    ROUND(AVG(VALIDATION_SCORE), 2)                               AS AVG_VALIDATION_SCORE,
    SUM(CASE WHEN STATUS = 'PROCESSED' THEN TOTAL_AMOUNT ELSE 0 END)
                                                                  AS TOTAL_PROCESSED_SPEND,
    SUM(CASE WHEN STATUS = 'REVIEW' THEN TOTAL_AMOUNT ELSE 0 END)
                                                                  AS TOTAL_PENDING_SPEND,
    ROUND(AVG(CASE WHEN APPROVED_TS IS NOT NULL
        THEN DATEDIFF('second', EXTRACTION_TS, APPROVED_TS)
        END), 0)                                                  AS AVG_PROCESSING_SECONDS
FROM INVOICE_HEADER;

CREATE OR REPLACE VIEW V_SPEND_ANALYSIS AS
SELECT
    h.PROPERTY,
    COALESCE(v.VENDOR_NAME, h.VENDOR_NAME_RAW) AS VENDOR_NAME,
    g.CATEGORY                                  AS GL_CATEGORY,
    g.GL_DESCRIPTION,
    COUNT(DISTINCT h.INVOICE_ID)                AS INVOICE_COUNT,
    SUM(li.LINE_TOTAL)                          AS TOTAL_SPEND,
    AVG(li.GL_CODE_CONFIDENCE)                  AS AVG_GL_CONFIDENCE
FROM INVOICE_HEADER h
JOIN INVOICE_LINE_ITEMS li
    ON h.INVOICE_ID = li.INVOICE_ID
LEFT JOIN VENDOR_MASTER v
    ON h.VENDOR_ID_RESOLVED = v.VENDOR_ID
LEFT JOIN GL_CODES g
    ON COALESCE(li.GL_CODE_CONFIRMED, li.GL_CODE_SUGGESTED) = g.GL_CODE
WHERE h.STATUS = 'PROCESSED'
GROUP BY h.PROPERTY, COALESCE(v.VENDOR_NAME, h.VENDOR_NAME_RAW), g.CATEGORY, g.GL_DESCRIPTION;

In [ ]:
SELECT * FROM V_PIPELINE_METRICS;

In [ ]:
SELECT * FROM V_SPEND_ANALYSIS
ORDER BY TOTAL_SPEND DESC
LIMIT 10;

## Step 5: Stream & Task Automation

The automation layer watches for new invoice extractions and routes them:

1. **Stream** — append-only stream on `INVOICE_HEADER` captures new rows
2. **SP_VALIDATE_AND_ROUTE** — scores each invoice; routes low scorers to `REVIEW_QUEUE`, auto-approves high scorers
3. **Task** — runs every 5 minutes when the stream has data (created suspended)
4. **SP_PROCESS_INVOICE** — end-to-end: AI_EXTRACT from PDF, vendor match, score, insert header

In [ ]:
CREATE STREAM IF NOT EXISTS INVOICE_HEADER_STREAM
    ON TABLE INVOICE_HEADER
    APPEND_ONLY = TRUE
    COMMENT = 'Captures new invoice extractions for validation pipeline';

In [ ]:
CREATE OR REPLACE PROCEDURE SP_VALIDATE_AND_ROUTE(THRESHOLD NUMBER)
RETURNS VARCHAR
LANGUAGE SQL
COMMENT = 'Validates new extractions and routes low-scoring invoices to review'
AS
$$
DECLARE
    routed_count NUMBER DEFAULT 0;
    approved_count NUMBER DEFAULT 0;
BEGIN
    INSERT INTO REVIEW_QUEUE (INVOICE_ID, FLAGGED_FIELDS, VALIDATION_SCORE)
    SELECT
        s.INVOICE_ID,
        ARRAY_CONSTRUCT_COMPACT(
            IFF(s.VENDOR_ID_RESOLVED IS NULL, 'VENDOR_ID_RESOLVED', NULL),
            IFF(s.INVOICE_NUMBER IS NULL, 'INVOICE_NUMBER', NULL),
            IFF(s.INVOICE_DATE IS NULL, 'INVOICE_DATE', NULL),
            IFF(s.TOTAL_AMOUNT IS NULL, 'TOTAL_AMOUNT', NULL),
            IFF(s.PO_REFERENCE IS NULL, 'PO_REFERENCE', NULL)
        ),
        s.VALIDATION_SCORE
    FROM INVOICE_HEADER_STREAM s
    WHERE s.VALIDATION_SCORE < :THRESHOLD
      AND s.STATUS = 'PENDING';

    routed_count := SQLROWCOUNT;

    UPDATE INVOICE_HEADER
    SET STATUS = 'PROCESSED',
        APPROVED_BY = 'SYSTEM',
        APPROVED_TS = CURRENT_TIMESTAMP()
    WHERE INVOICE_ID IN (
        SELECT INVOICE_ID
        FROM INVOICE_HEADER
        WHERE VALIDATION_SCORE >= :THRESHOLD
          AND STATUS = 'PENDING'
    );

    approved_count := SQLROWCOUNT;

    INSERT INTO AUDIT_LOG (INVOICE_ID, ACTION, FIELD_NAME, OLD_VALUE, NEW_VALUE, ACTOR, ACTOR_TYPE)
    SELECT INVOICE_ID, 'AUTO_APPROVED', 'STATUS', 'PENDING', 'PROCESSED', 'SYSTEM', 'SYSTEM'
    FROM INVOICE_HEADER
    WHERE APPROVED_BY = 'SYSTEM'
      AND APPROVED_TS >= DATEADD('minute', -1, CURRENT_TIMESTAMP());

    RETURN 'Validation complete: ' || approved_count || ' auto-approved, '
           || routed_count || ' routed to review';
END;
$$;

In [ ]:
CREATE TASK IF NOT EXISTS VALIDATE_INVOICES_TASK
    WAREHOUSE = SFE_AP_INVOICE_WH
    SCHEDULE = '5 MINUTE'
    COMMENT = 'Auto-validates new extractions using stream trigger'
    WHEN SYSTEM$STREAM_HAS_DATA('INVOICE_HEADER_STREAM')
AS
    CALL SP_VALIDATE_AND_ROUTE(0.75);

-- Task is created SUSPENDED by default. To start automated processing:
-- ALTER TASK VALIDATE_INVOICES_TASK RESUME;

In [ ]:
CREATE OR REPLACE PROCEDURE SP_PROCESS_INVOICE(FILE_PATH VARCHAR, PROPERTY VARCHAR)
RETURNS VARIANT
LANGUAGE SQL
COMMENT = 'End-to-end invoice processing — extract, match, classify, score'
AS
$$
DECLARE
    extraction VARIANT;
    vendor_name_raw VARCHAR;
    vendor_id_match NUMBER;
    invoice_num VARCHAR;
    invoice_dt DATE;
    po_ref VARCHAR;
    total_amt NUMBER(12,2);
    score NUMBER(5,2) DEFAULT 0;
    new_invoice_id NUMBER;
BEGIN
    -- AI_EXTRACT structured fields from PDF
    SELECT AI_EXTRACT(
        file => TO_FILE('@RAW_INVOICE_STAGE', :FILE_PATH),
        responseFormat => {
            'vendor_name': 'What is the vendor or supplier name on this invoice?',
            'invoice_number': 'What is the invoice number?',
            'invoice_date': 'What is the invoice date?',
            'po_reference': 'What is the purchase order or PO number?',
            'total_amount': 'What is the total amount due on this invoice?'
        }
    ) INTO extraction;

    vendor_name_raw := extraction:response:vendor_name::VARCHAR;
    invoice_num := extraction:response:invoice_number::VARCHAR;
    po_ref := extraction:response:po_reference::VARCHAR;
    total_amt := TRY_TO_NUMBER(REGEXP_REPLACE(extraction:response:total_amount::VARCHAR, '[^0-9.]', ''), 12, 2);

    BEGIN
        invoice_dt := TRY_TO_DATE(extraction:response:invoice_date::VARCHAR);
    EXCEPTION WHEN OTHER THEN
        invoice_dt := NULL;
    END;

    -- Fuzzy-match vendor against VENDOR_MASTER
    SELECT VENDOR_ID INTO vendor_id_match
    FROM VENDOR_MASTER
    WHERE VENDOR_NAME = :vendor_name_raw
       OR ARRAY_CONTAINS(:vendor_name_raw::VARIANT, VENDOR_ALIASES)
    QUALIFY ROW_NUMBER() OVER (ORDER BY JAROWINKLER_SIMILARITY(VENDOR_NAME, :vendor_name_raw) DESC) = 1;

    -- Compute validation score
    score := 0;
    IF (vendor_name_raw IS NOT NULL AND LENGTH(vendor_name_raw) > 0) THEN score := score + 0.15; END IF;
    IF (invoice_num IS NOT NULL AND LENGTH(invoice_num) > 2) THEN score := score + 0.15; END IF;
    IF (invoice_dt IS NOT NULL) THEN score := score + 0.15; END IF;
    IF (total_amt IS NOT NULL AND total_amt > 0) THEN score := score + 0.15; END IF;
    IF (po_ref IS NOT NULL AND LENGTH(po_ref) > 0) THEN score := score + 0.10; END IF;
    IF (vendor_id_match IS NOT NULL) THEN score := score + 0.20; END IF;
    IF (extraction:error IS NULL) THEN score := score + 0.10; END IF;

    -- Insert into INVOICE_HEADER
    INSERT INTO INVOICE_HEADER (
        SOURCE_FILE, VENDOR_NAME_RAW, VENDOR_ID_RESOLVED,
        INVOICE_NUMBER, INVOICE_DATE, PO_REFERENCE,
        TOTAL_AMOUNT, PROPERTY, VALIDATION_SCORE,
        STATUS, AI_EXTRACT_RAW
    ) VALUES (
        :FILE_PATH, :vendor_name_raw, :vendor_id_match,
        :invoice_num, :invoice_dt, :po_ref,
        :total_amt, :PROPERTY, :score,
        IFF(:score >= 0.75, 'PENDING', 'REVIEW'),
        :extraction
    );

    new_invoice_id := SQLROWCOUNT;

    -- Log to audit trail
    INSERT INTO AUDIT_LOG (INVOICE_ID, ACTION, ACTOR, ACTOR_TYPE)
    SELECT MAX(INVOICE_ID), 'EXTRACTED', 'AI_EXTRACT', 'AI'
    FROM INVOICE_HEADER;

    RETURN OBJECT_CONSTRUCT(
        'invoice_id', new_invoice_id,
        'vendor_raw', vendor_name_raw,
        'vendor_matched', vendor_id_match IS NOT NULL,
        'validation_score', score,
        'status', IFF(score >= 0.75, 'PENDING', 'REVIEW')
    );
END;
$$;

## Step 6: AI Extract Patterns

These cells demonstrate `AI_EXTRACT` for pulling structured data from PDF invoices.
**Upload a PDF** to `@RAW_INVOICE_STAGE` first, then update the filename and run the cell.

```sql
-- Upload a file to the stage (run from SnowSQL or Snowsight):
PUT file:///path/to/invoice.pdf @RAW_INVOICE_STAGE AUTO_COMPRESS=FALSE;
```

The demo ships with pre-populated sample data (Step 3), so these patterns are
optional — use them when processing your own invoices.

In [ ]:
-- Pattern 1: Extract header fields from a single invoice PDF
-- Update 'your_invoice.pdf' with the filename you uploaded to @RAW_INVOICE_STAGE

SELECT AI_EXTRACT(
    file => TO_FILE('@RAW_INVOICE_STAGE', 'your_invoice.pdf'),
    responseFormat => {
        'vendor_name': 'What is the vendor or supplier name?',
        'invoice_number': 'What is the invoice number or reference?',
        'invoice_date': 'What is the invoice date (YYYY-MM-DD)?',
        'po_reference': 'What is the purchase order or PO number?',
        'total_amount': 'What is the total amount due?',
        'currency': 'What currency is this invoice in (3-letter code)?'
    }
);

In [ ]:
-- Pattern 2: Extract line items as a structured table from a PDF

SELECT AI_EXTRACT(
    file => TO_FILE('@RAW_INVOICE_STAGE', 'your_invoice.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'line_items': {
                    'description': 'Invoice line items table',
                    'type': 'object',
                    'column_ordering': ['description', 'quantity', 'unit_price', 'total'],
                    'properties': {
                        'description': {'description': 'Item description', 'type': 'array'},
                        'quantity':    {'description': 'Quantity',         'type': 'array'},
                        'unit_price':  {'description': 'Unit Price',      'type': 'array'},
                        'total':       {'description': 'Line Total',      'type': 'array'}
                    }
                }
            }
        }
    }
);

In [ ]:
-- Pattern 3: Combined extraction (header + line items in one call)

SELECT AI_EXTRACT(
    file => TO_FILE('@RAW_INVOICE_STAGE', 'your_invoice.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'vendor_name': {
                    'description': 'Vendor or supplier company name',
                    'type': 'string'
                },
                'invoice_number': {
                    'description': 'Invoice number or reference ID',
                    'type': 'string'
                },
                'total_amount': {
                    'description': 'Total amount due on invoice',
                    'type': 'string'
                },
                'line_items': {
                    'description': 'Invoice line items',
                    'type': 'object',
                    'column_ordering': ['description', 'quantity', 'unit_price', 'total'],
                    'properties': {
                        'description': {'description': 'Item description', 'type': 'array'},
                        'quantity':    {'description': 'Quantity',         'type': 'array'},
                        'unit_price':  {'description': 'Unit Price',      'type': 'array'},
                        'total':       {'description': 'Line Total',      'type': 'array'}
                    }
                }
            }
        }
    }
);

In [ ]:
-- Pattern 4: Batch-process all PDFs on stage

SELECT
    relative_path AS file_name,
    AI_EXTRACT(
        file => TO_FILE('@RAW_INVOICE_STAGE', relative_path),
        responseFormat => {
            'vendor_name': 'Vendor or supplier name',
            'invoice_number': 'Invoice number',
            'invoice_date': 'Invoice date (YYYY-MM-DD)',
            'total_amount': 'Total amount due'
        }
    ) AS extracted_data
FROM DIRECTORY(@RAW_INVOICE_STAGE)
WHERE relative_path LIKE '%.pdf';

## Step 7: AI Classify GL Codes

Two procedures that use `AI_CLASSIFY` to suggest GL account codes for line-item descriptions.
The GL taxonomy from `GL_CODES` (loaded in Step 3) is passed as the set of candidate labels.

- **SP_CLASSIFY_LINE_ITEM** — classify a single line item by ID
- **SP_CLASSIFY_ALL_PENDING** — batch-classify all line items missing a GL suggestion

In [ ]:
CREATE OR REPLACE PROCEDURE SP_CLASSIFY_LINE_ITEM(LINE_ITEM_ID NUMBER)
RETURNS VARIANT
LANGUAGE SQL
COMMENT = 'Classifies a line item description into GL codes using AI_CLASSIFY'
AS
$$
DECLARE
    item_desc VARCHAR;
    gl_categories VARIANT;
    classification VARIANT;
    suggested_gl VARCHAR;
BEGIN
    SELECT DESCRIPTION INTO item_desc
    FROM INVOICE_LINE_ITEMS
    WHERE LINE_ID = :LINE_ITEM_ID;

    SELECT ARRAY_AGG(
        OBJECT_CONSTRUCT(
            'label', GL_CODE,
            'description', GL_DESCRIPTION || ' (' || CATEGORY || ')'
        )
    ) INTO gl_categories
    FROM GL_CODES;

    SELECT AI_CLASSIFY(
        :item_desc,
        :gl_categories,
        {
            'task_description': 'Classify this invoice line item into the correct GL account code for a gaming and hospitality company'
        }
    ) INTO classification;

    suggested_gl := classification:labels[0]::VARCHAR;

    UPDATE INVOICE_LINE_ITEMS
    SET GL_CODE_SUGGESTED = :suggested_gl
    WHERE LINE_ID = :LINE_ITEM_ID
      AND GL_CODE_CONFIRMED IS NULL;

    RETURN OBJECT_CONSTRUCT(
        'line_id', LINE_ITEM_ID,
        'description', item_desc,
        'suggested_gl', suggested_gl,
        'full_result', classification
    );
END;
$$;

CREATE OR REPLACE PROCEDURE SP_CLASSIFY_ALL_PENDING()
RETURNS VARCHAR
LANGUAGE SQL
COMMENT = 'Batch GL classification for all unclassified line items'
AS
$$
DECLARE
    classified_count NUMBER DEFAULT 0;
BEGIN
    LET gl_categories VARIANT := (
        SELECT ARRAY_AGG(
            OBJECT_CONSTRUCT(
                'label', GL_CODE,
                'description', GL_DESCRIPTION || ' (' || CATEGORY || ')'
            )
        ) FROM GL_CODES
    );

    MERGE INTO INVOICE_LINE_ITEMS tgt
    USING (
        SELECT
            li.LINE_ID,
            AI_CLASSIFY(
                li.DESCRIPTION,
                :gl_categories,
                {'task_description': 'Classify this invoice line item into the correct GL account code for a gaming and hospitality company'}
            ):labels[0]::VARCHAR AS SUGGESTED_GL
        FROM INVOICE_LINE_ITEMS li
        WHERE li.GL_CODE_SUGGESTED IS NULL
          AND li.GL_CODE_CONFIRMED IS NULL
          AND li.DESCRIPTION IS NOT NULL
    ) src
    ON tgt.LINE_ID = src.LINE_ID
    WHEN MATCHED THEN UPDATE SET
        tgt.GL_CODE_SUGGESTED = src.SUGGESTED_GL;

    classified_count := SQLROWCOUNT;

    RETURN 'Classified ' || classified_count || ' line items';
END;
$$;

## Step 8: Semantic View for Cortex Analyst

Creates a semantic view over the analytical views, enabling natural-language queries
like *"total spend by property this month"* via Cortex Analyst.

The semantic view defines:
- **Tables** — `PROCESSED_INVOICES`, `V_LINE_ITEMS_ENRICHED`, `V_SPEND_ANALYSIS` with synonyms
- **Facts** — amounts, scores, processing times
- **Dimensions** — property, vendor, status, GL codes
- **Metrics** — pre-defined aggregations (total spend, approval rate, etc.)
- **AI SQL hints** — guidance for the Analyst LLM on query intent

In [ ]:
CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS
  COMMENT = 'Shared schema for semantic views across demo projects';

CREATE OR REPLACE SEMANTIC VIEW SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS.SV_AP_INVOICE

    TABLES (
        inv AS SNOWFLAKE_EXAMPLE.AP_INVOICE.PROCESSED_INVOICES
            PRIMARY KEY (INVOICE_ID)
            WITH SYNONYMS = ('invoice', 'invoices', 'AP invoice', 'bill', 'bills')
            COMMENT = 'Processed AP invoices with vendor resolution and line item totals',

        lines AS SNOWFLAKE_EXAMPLE.AP_INVOICE.V_LINE_ITEMS_ENRICHED
            PRIMARY KEY (LINE_ID)
            WITH SYNONYMS = ('line item', 'line items', 'invoice detail', 'item')
            COMMENT = 'Invoice line items with GL classification status',

        spend AS SNOWFLAKE_EXAMPLE.AP_INVOICE.V_SPEND_ANALYSIS
            WITH SYNONYMS = ('spending', 'spend analysis', 'cost breakdown')
            COMMENT = 'Pre-aggregated spend by property, vendor, and GL category'
    )

    RELATIONSHIPS (
        inv_to_lines AS lines(INVOICE_ID) REFERENCES inv(INVOICE_ID)
    )

    FACTS (
        inv.TOTAL_AMOUNT  AS TOTAL_AMOUNT
            WITH SYNONYMS = ('amount', 'invoice total', 'total', 'invoice amount')
            COMMENT = 'Total invoice amount in the invoiced currency (typically USD)',

        inv.VALIDATION_SCORE  AS VALIDATION_SCORE
            WITH SYNONYMS = ('confidence', 'confidence score', 'quality score', 'extraction quality')
            COMMENT = 'Composite validation score (0.0-1.0) computed from field completeness, format checks, and vendor matching. Threshold for auto-approval is 0.75',

        inv.PROCESSING_SECONDS  AS PROCESSING_SECONDS
            WITH SYNONYMS = ('processing time', 'time to approve', 'cycle time')
            COMMENT = 'Seconds elapsed between AI extraction and approval (auto or human)',

        lines.LINE_AMOUNT  AS LINE_TOTAL
            WITH SYNONYMS = ('line total', 'item amount', 'line item total')
            COMMENT = 'Total amount for a single line item (quantity * unit price)',

        lines.QUANTITY  AS QUANTITY
            WITH SYNONYMS = ('qty', 'item quantity', 'units')
            COMMENT = 'Quantity of items on the line',

        lines.UNIT_PRICE  AS UNIT_PRICE
            WITH SYNONYMS = ('price', 'unit cost', 'per-unit price')
            COMMENT = 'Price per unit for the line item',

        lines.GL_CONFIDENCE  AS GL_CODE_CONFIDENCE
            WITH SYNONYMS = ('classification confidence', 'GL confidence')
            COMMENT = 'AI_CLASSIFY confidence score for the suggested GL code (0.0-1.0)',

        spend.CATEGORY_SPEND  AS TOTAL_SPEND
            WITH SYNONYMS = ('category total', 'category spend')
            COMMENT = 'Total spend for a property/vendor/GL category combination',

        spend.CATEGORY_INVOICE_COUNT  AS INVOICE_COUNT
            WITH SYNONYMS = ('number of invoices', 'invoice count per category')
            COMMENT = 'Count of distinct invoices in a property/vendor/GL category grouping'
    )

    DIMENSIONS (
        inv.INVOICE_NUMBER  AS INVOICE_NUMBER
            WITH SYNONYMS = ('invoice num', 'invoice #', 'inv number')
            COMMENT = 'Unique invoice identifier assigned by the vendor',

        inv.INVOICE_DATE  AS INVOICE_DATE
            WITH SYNONYMS = ('date', 'invoice dt', 'billing date')
            COMMENT = 'Date the invoice was issued by the vendor',

        inv.PROPERTY  AS PROPERTY
            WITH SYNONYMS = ('resort', 'location', 'site', 'facility')
            COMMENT = 'Property or resort that received the goods/services. Values: Resort East, Resort Northeast, Resort North',

        inv.STATUS  AS STATUS
            WITH SYNONYMS = ('invoice status', 'processing status', 'approval status')
            COMMENT = 'Current processing status: PROCESSED (auto-approved), REVIEW (needs human review), or PENDING',

        inv.VENDOR_NAME  AS VENDOR_NAME_RESOLVED
            WITH SYNONYMS = ('vendor', 'supplier', 'company name', 'vendor name')
            COMMENT = 'Canonical vendor name after fuzzy matching against vendor master',

        inv.PO_REFERENCE  AS PO_REFERENCE
            WITH SYNONYMS = ('PO', 'purchase order', 'PO number', 'PO #')
            COMMENT = 'Associated purchase order reference number',

        inv.CURRENCY  AS CURRENCY
            WITH SYNONYMS = ('currency code')
            COMMENT = 'Invoice currency (3-letter ISO code, typically USD)',

        lines.LINE_DESCRIPTION  AS DESCRIPTION
            WITH SYNONYMS = ('item description', 'line item description', 'item')
            COMMENT = 'Description of the goods or services on a line item',

        lines.GL_CODE  AS GL_CODE_SUGGESTED
            WITH SYNONYMS = ('GL', 'GL account', 'account code', 'general ledger code')
            COMMENT = 'GL account code suggested by AI_CLASSIFY for the line item',

        lines.GL_DESCRIPTION  AS GL_SUGGESTED_DESC
            WITH SYNONYMS = ('GL name', 'account description', 'GL category')
            COMMENT = 'Human-readable description of the suggested GL code',

        lines.GL_CLASSIFICATION_STATUS  AS CLASSIFICATION_STATUS
            WITH SYNONYMS = ('classification status', 'GL status')
            COMMENT = 'Whether the GL code was AI-suggested, human-confirmed, or needs review',

        spend.SPEND_CATEGORY  AS GL_CATEGORY
            WITH SYNONYMS = ('expense category', 'cost category', 'spend type')
            COMMENT = 'High-level expense category: Operating or G&A'
    )

    METRICS (
        inv.TOTAL_INVOICE_SPEND AS SUM(inv.TOTAL_AMOUNT)
            WITH SYNONYMS = ('total spend', 'total AP spend', 'aggregate spend')
            COMMENT = 'Sum of all invoice amounts across the selected scope',

        inv.AVERAGE_INVOICE_AMOUNT AS AVG(inv.TOTAL_AMOUNT)
            WITH SYNONYMS = ('avg invoice', 'average bill', 'mean invoice amount')
            COMMENT = 'Average invoice amount across the selected scope',

        inv.INVOICE_COUNT AS COUNT(inv.INVOICE_ID)
            WITH SYNONYMS = ('number of invoices', 'how many invoices', 'invoice count')
            COMMENT = 'Total count of invoices in the selected scope',

        inv.AUTO_APPROVAL_RATE AS
            ROUND(COUNT_IF(inv.STATUS = 'PROCESSED') * 100.0 / NULLIF(COUNT(inv.INVOICE_ID), 0), 1)
            WITH SYNONYMS = ('approval rate', 'automation rate', 'straight-through rate')
            COMMENT = 'Percentage of invoices auto-approved without human intervention',

        inv.AVERAGE_VALIDATION_SCORE AS ROUND(AVG(inv.VALIDATION_SCORE), 2)
            WITH SYNONYMS = ('avg confidence', 'mean quality score')
            COMMENT = 'Average extraction validation score across selected invoices',

        inv.AVERAGE_PROCESSING_TIME AS ROUND(AVG(inv.PROCESSING_SECONDS), 0)
            WITH SYNONYMS = ('avg cycle time', 'mean processing time')
            COMMENT = 'Average seconds from extraction to approval',

        inv.PENDING_REVIEW_COUNT AS COUNT_IF(inv.STATUS = 'REVIEW')
            WITH SYNONYMS = ('pending count', 'review queue size', 'exceptions')
            COMMENT = 'Number of invoices currently pending human review',

        lines.TOTAL_LINE_SPEND AS SUM(lines.LINE_AMOUNT)
            WITH SYNONYMS = ('total line item spend', 'aggregate line spend')
            COMMENT = 'Sum of all line item amounts across selected scope'
    )

    COMMENT = 'AP Invoice Pipeline analytics for Cortex Analyst NL queries'

    AI_SQL_GENERATION '
        When a user asks about "spend" or "total spend", use TOTAL_INVOICE_SPEND metric on the inv table.
        When a user asks about "pending" or "review" invoices, filter by STATUS = ''REVIEW''.
        When a user asks about a specific property, filter PROPERTY dimension.
        "Approval rate" or "automation rate" refers to the AUTO_APPROVAL_RATE metric.
        "Processing time" refers to AVERAGE_PROCESSING_TIME in seconds; convert to minutes or hours for display.
        For vendor-level analysis, use VENDOR_NAME dimension.
        For GL-level analysis, use GL_CODE and GL_DESCRIPTION dimensions.
        Default time range is current month unless specified otherwise.
    ';

In [ ]:
-- Try a natural-language query against the semantic view
SELECT * FROM SEMANTIC_VIEW(
    SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS.SV_AP_INVOICE,
    QUESTION => 'What is the total spend by property?'
);

## Step 9: Inline Analytics

Pipeline KPIs, ROI comparison, and spend analysis rendered directly in the notebook
using Snowpark + pandas. This replicates the read-only analytics from the
Streamlit dashboard without requiring the full app deployment.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
SCHEMA = 'SNOWFLAKE_EXAMPLE.AP_INVOICE'

metrics = session.sql(f'''
    SELECT
        COUNT(*)                                         AS total_invoices,
        COUNT_IF(STATUS = 'PROCESSED')                   AS processed,
        COUNT_IF(STATUS = 'REVIEW')                      AS in_review,
        COUNT_IF(STATUS = 'PENDING')                     AS pending,
        ROUND(AVG(VALIDATION_SCORE), 2)                  AS avg_score,
        SUM(CASE WHEN STATUS = 'PROCESSED' THEN TOTAL_AMOUNT ELSE 0 END) AS processed_spend,
        SUM(CASE WHEN STATUS = 'REVIEW' THEN TOTAL_AMOUNT ELSE 0 END)    AS review_spend,
        ROUND(AVG(CASE WHEN APPROVED_TS IS NOT NULL
            THEN DATEDIFF('second', EXTRACTION_TS, APPROVED_TS) END), 0) AS avg_proc_sec
    FROM {SCHEMA}.INVOICE_HEADER
''').collect()[0]

total = metrics['TOTAL_INVOICES']
processed = metrics['PROCESSED']
in_review = metrics['IN_REVIEW']
avg_score = metrics['AVG_SCORE']
processed_spend = metrics['PROCESSED_SPEND']
review_spend = metrics['REVIEW_SPEND']
avg_sec = metrics['AVG_PROC_SEC'] or 0

kpis = pd.DataFrame([
    {'Metric': 'Total Invoices',       'Value': f'{total}'},
    {'Metric': 'Auto-Approved',        'Value': f'{processed} ({round(processed / max(total, 1) * 100, 1)}%)'},
    {'Metric': 'In Review',            'Value': f'{in_review}'},
    {'Metric': 'Avg Validation Score', 'Value': f'{avg_score}'},
    {'Metric': 'Processed Spend',      'Value': f'${processed_spend:,.2f}'},
    {'Metric': 'Pending Review Spend', 'Value': f'${review_spend:,.2f}'},
    {'Metric': 'Avg Processing Time',  'Value': f'{avg_sec}s'},
])
print('Pipeline KPIs')
print('=' * 50)
display(kpis)

manual_min = 12
auto_sec = avg_sec if avg_sec > 0 else 5
savings = max(round((1 - (auto_sec / 60) / manual_min) * 100, 1), 0)

roi = pd.DataFrame([
    {'Metric': 'Manual Processing (est.)', 'Value': f'{total * manual_min} min'},
    {'Metric': 'Automated Processing',     'Value': f'{round(total * auto_sec / 60, 1)} min'},
    {'Metric': 'Time Savings',             'Value': f'{savings}%'},
])
print('\nROI: Manual vs. Automated')
print('=' * 50)
display(roi)

In [ ]:
from snowflake.snowpark.context import get_active_session
import matplotlib.pyplot as plt

session = get_active_session()
SCHEMA = 'SNOWFLAKE_EXAMPLE.AP_INVOICE'

spend_by_prop = session.sql(f'''
    SELECT PROPERTY, COUNT(*) AS invoice_count, SUM(TOTAL_AMOUNT) AS total_spend
    FROM {SCHEMA}.INVOICE_HEADER
    WHERE STATUS = 'PROCESSED'
    GROUP BY PROPERTY
    ORDER BY total_spend DESC
''').to_pandas()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(spend_by_prop['PROPERTY'], spend_by_prop['TOTAL_SPEND'], color='#29B5E8')
ax.bar_label(bars, fmt='$%,.0f', padding=4)
ax.set_xlabel('Total Spend ($)')
ax.set_title('Spend by Property')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

print('\nTop Vendors by Spend')
print('=' * 50)
top_vendors = session.sql(f'''
    SELECT
        COALESCE(v.VENDOR_NAME, h.VENDOR_NAME_RAW) AS vendor,
        COUNT(DISTINCT h.INVOICE_ID) AS invoices,
        SUM(h.TOTAL_AMOUNT) AS total_spend
    FROM {SCHEMA}.INVOICE_HEADER h
    LEFT JOIN {SCHEMA}.VENDOR_MASTER v ON h.VENDOR_ID_RESOLVED = v.VENDOR_ID
    WHERE h.STATUS = 'PROCESSED'
    GROUP BY vendor
    ORDER BY total_spend DESC
    LIMIT 10
''').to_pandas()
display(top_vendors)

## Step 10: Deploy Streamlit Dashboard

The interactive Streamlit dashboard provides capabilities beyond what a notebook can offer:
- **Review Queue** with approve/reject buttons that write back to the database
- **Analytics Chat** powered by Cortex Analyst for natural-language questions
- **Sidebar navigation** between Pipeline Status, Review Queue, and Analytics Chat

This cell sets up the Git repository connection and deploys the app from source.

In [ ]:
CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_EXAMPLE.GIT_REPOS
  COMMENT = 'Shared schema for Git repository stages across demo projects';

CREATE GIT REPOSITORY IF NOT EXISTS SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_DEMOS_REPO
  API_INTEGRATION = SFE_GIT_API_INTEGRATION
  ORIGIN = 'https://github.com/sfc-gh-miwhitaker/sfe-public.git'
  COMMENT = 'Shared monorepo Git repository';

ALTER GIT REPOSITORY SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_DEMOS_REPO FETCH;

CREATE OR REPLACE STREAMLIT SNOWFLAKE_EXAMPLE.AP_INVOICE.AP_INVOICE_DASHBOARD
    FROM '@SNOWFLAKE_EXAMPLE.GIT_REPOS.SFE_DEMOS_REPO/branches/main/demo-ap-invoice/streamlit'
    MAIN_FILE = 'streamlit_app.py'
    QUERY_WAREHOUSE = SFE_AP_INVOICE_WH
    TITLE = 'AP Invoice Pipeline'
    COMMENT = '3-panel AP invoice dashboard — status, review queue, analytics chat';

ALTER STREAMLIT SNOWFLAKE_EXAMPLE.AP_INVOICE.AP_INVOICE_DASHBOARD ADD LIVE VERSION FROM LAST;

SHOW STREAMLITS LIKE 'AP_INVOICE_DASHBOARD' IN SCHEMA SNOWFLAKE_EXAMPLE.AP_INVOICE;

## Teardown

Run this cell to remove all AP Invoice Pipeline objects. The shared database
(`SNOWFLAKE_EXAMPLE`), Git schema, and API integration are preserved.

In [ ]:
USE ROLE SYSADMIN;

DROP STREAMLIT IF EXISTS SNOWFLAKE_EXAMPLE.AP_INVOICE.AP_INVOICE_DASHBOARD;
DROP TASK IF EXISTS SNOWFLAKE_EXAMPLE.AP_INVOICE.VALIDATE_INVOICES_TASK;
DROP STREAM IF EXISTS SNOWFLAKE_EXAMPLE.AP_INVOICE.INVOICE_HEADER_STREAM;
DROP SCHEMA IF EXISTS SNOWFLAKE_EXAMPLE.AP_INVOICE CASCADE;
DROP WAREHOUSE IF EXISTS SFE_AP_INVOICE_WH;
DROP SEMANTIC VIEW IF EXISTS SNOWFLAKE_EXAMPLE.SEMANTIC_MODELS.SV_AP_INVOICE;

SELECT 'Teardown complete' AS status;